In [1]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 문서 불러오기

In [2]:
file_path = "../../data/CN7N_2026_ko_KR.pdf"

loader = PyPDFLoader(file_path)

docs = loader.load()
len(docs)

436

In [3]:
docs[3].page_content

'이 자동차에는 사고기록장치가 장착되어 있습니다.\n사고기록장치는 자동차의 충돌 등 사고 전후 일정시간 \n동안 자동차의 운행 정보 (주행속도, 제동페달, 가속페\n달 등의 작동 여부)를 저장하고, 저장된 정보를 확인할 \n수 있는 기능을 하는 장치를 말합니다.\n사고기록정보는 사고 상황을 좀 더 잘 이해하는데 도움\n이 됩니다.\n사고기록장치 세부 \n안내문\n(제30조의3제1항 관련)'

## 문서 청킹

In [11]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = splitter.split_documents(docs)
len(chunks)

1327

In [12]:
chunks[3].page_content

'제작결함 안내\n제작결함 안내 \n(제50조 관련)\n자동차제작자등(부품제작자등) : 현대자동차(주)\n주 소 : 서울특별시 서초구 헌릉로 12(양재동)\n연락처 : 080-600-6000\n귀하의 자동차 또는 자동차부품에 잦은 고장 등의 문제\n로 교통사고를 유발할 수 있는 결함이 있다고 판단되면, \n자기 및 다른 사람의 안전을 위하여 즉시 현대자동차(주)\n와 제작결함조사를 시행하는 한국교통안전공단 자동차\n안전연구원에 연락하여 주시기 바랍니다.\n한국교통안전공단 자동차안전연구원은 소비자 불만사\n항 등을 접수하여 분석한 후 해당 자동차 또는 자동차부\n품에 제작결함의 가능성이 있다고 판단되는 경우 제작\n결함조사를 실시하여 해당 제작사에게 제작결함시정\n(recall) 등의 조치를 취할 것입니다.'

## 벡터스토어에 저장

In [13]:
persist_directory = "../7_vectorstore/chroma_store"
collection_name = "alert_for_car_damage"
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory=persist_directory,
    collection_name=collection_name
)


## 검색기 구성

In [16]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 5}
)

result = retriever.invoke("에어백 관련 주의 사항을 알고싶어.")
result

[Document(metadata={'creator': 'PyPDF', 'total_pages': 436, 'page_label': '2-1', 'producer': 'PyPDF', 'page': 18, 'source': '../../data/CN7N_2026_ko_KR.pdf', 'moddate': '2025-06-17T12:11:08+09:00', 'creationdate': '2025-06-17T12:11:08+09:00'}, page_content='안전벨트 착용 ..................................................2-3\n탑승자를 보호하는 안전벨트 착용 .................2-3\n기타 주의사항 ...............................................2-4\n에어백 관련 주의사항 .......................................2-4\n에어백은 보조 안전장치입니다. .....................2-4\n유아/어린이의 에어백 관련 주의사항 ............2-5\n주·정차 중 차내 수면 금지 .................................2-5'),
 Document(metadata={'creator': 'PyPDF', 'total_pages': 436, 'moddate': '2025-06-17T12:11:08+09:00', 'page': 53, 'creationdate': '2025-06-17T12:11:08+09:00', 'producer': 'PyPDF', 'source': '../../data/CN7N_2026_ko_KR.pdf', 'page_label': '3-26'}, page_content='경 고\n에어백이 작동할 때, 스티어링 휠과 크래쉬 패드의 에어\n백 관련 부품이 매우 뜨겁습니다. 부상을 방지하기 위해 \n에어백이 부풀어 오른 후에 에어백 내장 부분을 바로 만\n지지 마십시오.\n찰과상찰과상\n타박상타박상\n